In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
sys.path.append("../src")

from config import DATA_PATH, SCENARIOS_PATH 
from choice_model import scale_attribute 
from load_data import load_scenarios, load_beta_partworths_samples, load_beta_partworths_mean

In [3]:
beta_means = load_beta_partworths_mean(str(DATA_PATH))

In [4]:
scenarios_data = load_scenarios(str(SCENARIOS_PATH))
scenarios_data

,scenario,cost_change_pct_vs_ep2050,fuel_imports_twh,gross_winter_imports_twh,buildings_heat_electrification_pct,materials_land_concern,public_acceptance_note,land_use_km2_approx,land_use_precision,source
0,EP2050+,0.0,9.1,24.7,36.3,n.a.,5% for electricity imports,67.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
1,EP2050+ (5TWh constraint),64.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Table 1 and main text (main text); not reporte...
2,S/CCGT (Fossil) + DACCS,-16.3,31.8,25.3,100.0,NaN,25% for natural gas,67.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
3,S/CCGT (Ren. Methane) + H2,-3.6,0.0,24.4,100.0,NaN,66% for biomass,67.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
4,S/CCGT (Ren. Methane),-3.6,0.0,24.4,100.0,NaN,66% for biomass,67.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
5,Alpine PV,-0.8,0.0,24.0,56.4,"Land use, Graphite",56% for open-field PV,128.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
6,Wind,18.0,16.3,22.7,32.4,PGM,60%,72.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
7,Roof PV++,33.7,17.8,22.5,32.4,"Graphite, PGM",90%,72.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
8,CHP,22.2,17.4,23.2,33.5,NaN,NaN,72.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)
9,H2 Imports,19.2,18.7,22.3,3.0,PGM,NaN,67.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)


In [5]:
scenario_data_alpine_pv = scenarios_data[scenarios_data["scenario"] == "Alpine PV"]
scenario_data_alpine_pv

,scenario,cost_change_pct_vs_ep2050,fuel_imports_twh,gross_winter_imports_twh,buildings_heat_electrification_pct,materials_land_concern,public_acceptance_note,land_use_km2_approx,land_use_precision,source
5,Alpine PV,-0.8,0.0,24.0,56.4,"Land use, Graphite",56% for open-field PV,128.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)


In [6]:
SWISS_LAND_AREA_KM2 = 41_285  # total land area of Switzerland

land_km2 = scenario_data_alpine_pv["land_use_km2_approx"].iloc[0]
land_share = land_km2 / SWISS_LAND_AREA_KM2
land_share

np.float64(0.003100399660893787)

In [10]:
land_scaled = scale_attribute(land_share, "LAND")
land_scaled

np.float64(0.038754995761172334)

In [14]:
price_raw_decimal = scenario_data_alpine_pv["cost_change_pct_vs_ep2050"].iloc[0] / 100
price_scaled = scale_attribute(price_raw_decimal, "PRICES")

price_scaled

np.float64(-0.013333333333333334)

In [15]:
share_imports = 0 #placeholder for now, since i will need acctual data

In [16]:
x_alpine_pv = {
    "TECHNOLOGY:Open-field PV": 1,
    "TECHNOLOGY:Wind":0, 
    "TRANSMISSION":0,
    "LAND": land_scaled,
    "SHARE_IMPORTS": share_imports, 
    "PRICES": price_scaled,
}
x_alpine_pv

{'TECHNOLOGY:Open-field PV': 1,
 'TECHNOLOGY:Wind': 0,
 'TRANSMISSION': 0,
 'LAND': np.float64(0.038754995761172334),
 'SHARE_IMPORTS': 0,
 'PRICES': np.float64(-0.013333333333333334)}

In [18]:
beta_means

,attribute,beta_mean
0,TECHNOLOGY:Open-field PV,-0.198433
1,TECHNOLOGY:Wind,-0.432109
2,TRANSMISSION,-0.105768
3,LAND,-0.094972
4,SHARE_IMPORTS,-0.654340
5,PRICES,-1.163388


In [19]:
utility_alpine_pv = sum(x_alpine_pv[attr] * beta_means[beta_means["attribute"] == attr]["beta_mean"].values[0] for attr in x_alpine_pv)
utility_alpine_pv

np.float64(-0.18660205858649836)

In [20]:
for attr in x_alpine_pv:
    beta_val = beta_means[beta_means["attribute"] == attr]["beta_mean"].values[0]
    contribution = x_alpine_pv[attr] * beta_val
    print(f"{attr}: x={x_alpine_pv[attr]:.4f}, beta={beta_val:.4f}, contribution={contribution:.4f}")
    

TECHNOLOGY:Open-field PV: x=1.0000, beta=-0.1984, contribution=-0.1984
TECHNOLOGY:Wind: x=0.0000, beta=-0.4321, contribution=-0.0000
TRANSMISSION: x=0.0000, beta=-0.1058, contribution=-0.0000
LAND: x=0.0388, beta=-0.0950, contribution=-0.0037
SHARE_IMPORTS: x=0.0000, beta=-0.6543, contribution=-0.0000
PRICES: x=-0.0133, beta=-1.1634, contribution=0.0155


In [21]:
scenario_data_ep2050 = scenarios_data[scenarios_data["scenario"] == "EP2050+"]
scenario_data_ep2050

,scenario,cost_change_pct_vs_ep2050,fuel_imports_twh,gross_winter_imports_twh,buildings_heat_electrification_pct,materials_land_concern,public_acceptance_note,land_use_km2_approx,land_use_precision,source
0,EP2050+,0.0,9.1,24.7,36.3,n.a.,5% for electricity imports,67.0,approximate (read from Figure 1S bar chart),Table 3 (main text); Figure 1S (supplementary)


In [22]:
land_share_ep2050 = scenario_data_ep2050["land_use_km2_approx"].iloc[0] / SWISS_LAND_AREA_KM2
land_scaled_ep2050 = scale_attribute(land_share_ep2050, "LAND")
land_scaled_ep2050

np.float64(0.020285818093738646)

In [23]:
price_raw_decimal_ep2050 = scenario_data_ep2050["cost_change_pct_vs_ep2050"].iloc[0] / 100
price_scaled_ep2050 = scale_attribute(price_raw_decimal_ep2050, "PRICES")
price_scaled_ep2050

np.float64(0.0)

In [25]:
share_imports_ep2050 = 0 #placeholder for now, since i will need acctual data

In [26]:
x_ep2050 = {
    "TECHNOLOGY:Open-field PV": 0,
    "TECHNOLOGY:Wind":0,
    "TRANSMISSION":0,
    "LAND": land_scaled_ep2050,
    "SHARE_IMPORTS": share_imports_ep2050,
    "PRICES": price_scaled_ep2050,
}
x_ep2050

{'TECHNOLOGY:Open-field PV': 0,
 'TECHNOLOGY:Wind': 0,
 'TRANSMISSION': 0,
 'LAND': np.float64(0.020285818093738646),
 'SHARE_IMPORTS': 0,
 'PRICES': np.float64(0.0)}

In [27]:
utility_ep2050 = sum(x_ep2050[attr] * beta_means[beta_means["attribute"] == attr]["beta_mean"].values[0] for attr in x_ep2050)
utility_ep2050

np.float64(-0.0019265771720467296)